# HA: The Lens Problem

**Your name:**
**Date:**

## Background

The MadeUP Lab studies cell biology by imaging and analyzing cell populations. The lab's core workflow: researchers image cells, run them through an automated classifier, and use predictions to flag cells of interest.

For years the lab relied on a binary classifier that labels individual cells as dividing (mitotic) or non-dividing (non-mitotic).
The classifier was trained on images captured with the lab's original microscope lens and has performed reliably.

Last month the lab acquired a new microscope lens with different optical characteristics. The current experiment (experiment 5) was imaged with the new lens. When the biologists ran these images through the classifier, something was off. 
The biologists complained: 
*"The model doesn't work with the new lens."*

## Your task

You are a data scientist brought in to investigate. Your job is threefold:

1. Determine whether the biologists' complaint is justified and gather quantitative evidence.
2. Fix the model so it works on both old-lens and new-lens images.
3. Two new lenses should arrive next month - how can we flag problems with the classifier even before the
   biologists complain?

## Data and resources (cloned with the repository below)

* `experiments.zip` - 100 cell images per folder (unzipped to `experiments/` by the setup cells):
   * `current_experiment/5_new_lens` - experiment 5 captured with the new lens. These are the images the biologists
    ran through the classifier.

  * `old_experiments/1_old_lens` ... `4_old_lens` - four earlier experiments captured with the original microscope
    lens. These images represent the distribution of cells from the current experiment.
 
* `classifier.pt` - a pre-trained PyTorch model. The model takes a single-channel fluorescence image (DAPI/Hoechst
  staining) and returns a label (1 = dividing, 0 = non-dividing).
* `model.py` - the classifier class. Besides the label (`is_mitotic`) it exposes `get_logits` and `get_embeddings`
  (the features before the classification head).

## Guidelines

* `notebook.ipynb` - this notebook, with all code, plots, and written analysis. Cells must run from top to bottom
  without errors. All plots must be embedded in the notebook output (not saved as separate files).
  Add explanations and ideas in markdown cells in the notebook.

* You must be able to explain any code you submit.You will be asked to explain your approach and answer follow-up questions.
The notebook works in Google Colab (optionally **Runtime -> Change runtime type -> T4 GPU**) or locally.

## 0. Setup

In [ ]:
# Install the packages this notebook uses (preinstalled in Google Colab - then this is a quick no-op).
%pip install -q numpy pandas pillow matplotlib scikit-learn torch torchvision tqdm

In [ ]:
# Get the repository: model.py, classifier.pt, experiments.zip.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/lionben89/ZaritskyLab-HA.git"

if Path("model.py").exists():                 # already running inside a clone
    REPO_DIR = Path(".").resolve()
else:
    REPO_DIR = Path("ZaritskyLab-HA").resolve()
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_DIR = REPO_DIR
for item in ("model.py", "classifier.pt", "experiments.zip"):
    print(f"{item:14s}", "ok" if (DATA_DIR / item).exists() else "MISSING")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from model import ResNetClassifier, load_classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

### Loading the images

The next cell unzips `experiments.zip` and loads every experiment folder into two dictionaries keyed by the
experiment name (`"1_old_lens"` ... `"4_old_lens"`, `"5_new_lens"`, `"5_old_lens"`):

* `images[name]` - a float tensor `[100, 1, 96, 96]` (images, channel, height, width).
* `image_names[name]` - the file name of each image, in the same order.

The images are already preprocessed and ready to be used in the classifier: single-channel, pixel values in
`[0, 1]`, which is the input format `model.py` expects (resizing and normalisation happen inside the model).


In [ ]:
import zipfile

EXP_DIR = DATA_DIR / "experiments"
if not EXP_DIR.exists():
    with zipfile.ZipFile(DATA_DIR / "experiments.zip") as z:
        z.extractall(DATA_DIR)


def load_folder(folder: Path) -> tuple[torch.Tensor, list[str]]:
    """All PNGs of a folder as a float tensor [N, 1, H, W] with pixels in [0, 1], and their file names."""
    files = sorted(folder.glob("*.png"))
    imgs = [torch.from_numpy(np.asarray(Image.open(f), dtype=np.float32) / 65535)[None] for f in files]  # 16-bit
    return torch.stack(imgs), [f.name for f in files]


images, image_names = {}, {}
for folder in sorted(d for d in EXP_DIR.glob("*/*") if d.is_dir()):  # experiments/<group>/<experiment name>
    images[folder.name], image_names[folder.name] = load_folder(folder)
    print(f"{folder.parent.name:22s} {folder.name:12s} {tuple(images[folder.name].shape)}")

## 1. Is the complaint justified?

Gather quantitative evidence on whether the classifier behaves differently on new-lens images than on old-lens images.
Explain each result in terms a biologist would understand.

for that use only old_experiments and current_experiment

In [ ]:
# Your analysis here

**Findings:**

*Write your answer here.*

### Follow-up: experiment 5 imaged again with the original lens

After you showed that the biologists' complaint is justified, you asked the lab to image the cells of
experiment 5 again, this time with the original lens. These images are in
`followup_experiments/5_old_lens`: the same 100 cells as `current_experiment/5_new_lens`, captured with the
original lens.

Use these images in the next sections.


## 2. Fix the model so it works on both lenses

In [ ]:
# Your approach here

**What you did and why, and how you know it works:**

*Write your answer here.*

## 3. Two new lenses arrive next month - how do we flag problems early?

Propose (optionally - demonstrate on the data you have) a way to detect that the classifier is in trouble
*before* the biologists complain.

In [ ]:
# Your monitoring approach here

**Proposed procedure:**

*Write your answer here.*

## 4. Conclusions

*Summarise your the main limitations of your analysis.*
*Fill free to add whatever you think is important*